In [1]:
%load_ext autoreload
%autoreload 2

# Import

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Union
from glob import glob

import sys
PHASE1_SRC = Path("../Phase1/src/")
if str(PHASE1_SRC) not in sys.path:
    sys.path.insert(0, str(PHASE1_SRC))

from space import Space, GFLOWNET_ENV
from reward import reward_peak, reward_latent
from machinelearning import train_single_model, predict_with_model, preprocess, build_models
from VEM import oracle_fn, TARGET_NAMES
from plotting import *

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from umap import UMAP
from sklearn.decomposition import PCA


# Loading

In [2]:
run = "grid_results_2"

In [3]:
grid = pd.read_csv(f"{run}/grid_summary.csv")#.sort_values("run")

In [4]:
grid.head()

,run,status,elapsed_sec,best_reward,mean_reward,total_oracle_budget,sampling_strategy,acquisition,seed,n_init,n_candidates_per_iter,n_iterations
0,run_0,ok,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15
1,run_1,ok,6.8,0.984545,0.803565,350,random,top_k,123,200,10,15
2,run_2,ok,6.5,0.971533,0.763538,350,random,top_k,123,275,5,15
3,run_3,ok,5.9,0.996059,0.861917,350,random,top_k,456,50,20,15
4,run_4,ok,6.6,1.000000,0.804350,350,random,top_k,456,200,10,15


In [5]:
# grid["run"] = grid["run"].str[:7]
# grid["run"] = grid["run"].str.replace("run", "run_")
# grid["run"] = grid["run"].str.replace(r"run_0+(\d+)", r"run_\1", regex=True)

In [6]:
# grid.to_csv(f"{run}/grid_summary.csv", index=False)

In [7]:
from glob import glob
paths = sorted(glob(f"{run}/*/al_metrics.csv"))
if not paths:
    raise ValueError("No files found at al_results/*/al_metrics.csv")
df = pd.concat([load(p).assign(run=Path(p).parent.name) for p in paths], ignore_index=True)

In [8]:
df

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,best_reward_this_iter,mean_reward_this_iter,best_predicted_reward,elapsed_sec,run
0,1,17:17:16,70,70,20,0.953214,0.773339,0.208058,0.094107,random,0.953214,0.892436,0.890691,6.6,run_0
1,2,17:17:18,90,90,20,0.953214,0.797323,0.569753,0.099253,random,0.951304,0.881267,0.908221,2.5,run_0
2,3,17:17:19,110,110,20,0.968947,0.813842,0.675596,0.071394,random,0.968947,0.888174,0.908726,0.4,run_0
3,4,17:17:19,130,130,20,0.968947,0.828889,0.718318,0.060543,random,0.950932,0.911648,0.938576,0.4,run_0
4,5,17:17:20,150,150,20,0.968947,0.838112,0.627701,0.062788,random,0.962289,0.898063,0.943488,0.4,run_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1615,11,17:18:20,270,270,20,0.969298,0.852329,0.729605,0.063120,random,0.969298,0.908095,0.934479,0.4,run_9
1616,12,17:18:21,290,290,20,0.969298,0.855455,0.754186,0.056059,random,0.967568,0.897662,0.932681,0.4,run_9
1617,13,17:18:21,310,310,20,0.969298,0.857634,0.722797,0.060438,random,0.955636,0.889232,0.935670,0.4,run_9
1618,14,17:18:22,330,330,20,0.969298,0.861843,0.680453,0.059252,random,0.964353,0.927072,0.952723,0.5,run_9


In [9]:
df[df["sampling_strategy"]=="gflownet"].groupby("run").count().count().iloc[0]

np.int64(18)

In [10]:
best_runs(df)

,run,final_mean_reward,final_best_reward,final_proxy_r2,final_proxy_mae,total_oracle_evals,sampling_strategy
0,run_102,0.932578,1.000000,0.761743,0.034207,350,genetic
1,run_54,0.868662,1.000000,0.735829,0.050436,350,gflownet
2,run_76,0.815430,1.000000,0.783871,0.052682,350,gp
3,run_39,0.918791,1.000000,0.489636,0.041866,350,grid
4,run_33,0.864254,0.999497,0.697583,0.065675,350,lhs
5,run_13,0.798122,1.000000,0.806302,0.061732,350,random


## multi-curves (old)

In [ ]:

def plot_run(path: Union[str, Path], title: str = None) -> go.Figure:
    """
    Dashboard for one AL run.

    Subplots
    --------
    Row 1 left  : best reward so far (cumulative max) + this-iter best
    Row 1 right : mean reward so far + this-iter mean
    Row 2 left  : proxy R² over iterations
    Row 2 right : proxy MAE over iterations
    Row 3 left  : oracle evaluations cumulative
    Row 3 right : elapsed time per iteration (sec)
    """
    df = _load(path)
    run = title or Path(path).parent.name

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            "Best reward (cumulative)", "Mean reward",
            "Proxy R²", "Proxy MAE",
            "Oracle evaluations (cumulative)", "Time per iteration (s)",
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.10,
    )

    it = df["iteration"]
    blue, orange = "#4C72B0", "#DD8452"

    # --- Row 1: rewards ---
    fig.add_trace(go.Scatter(
        x=it, y=df["best_reward_so_far"],
        name="Best so far", line=dict(color=blue, width=2),
        mode="lines+markers", marker=dict(size=5),
    ), row=1, col=1)

    if "best_reward_this_iter" in df:
        fig.add_trace(go.Bar(
            x=it, y=df["best_reward_this_iter"],
            name="Best this iter", marker_color=orange, opacity=0.5,
        ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=it, y=df["mean_reward_so_far"],
        name="Mean so far", line=dict(color=blue, width=2),
        mode="lines+markers", marker=dict(size=5), showlegend=False,
    ), row=1, col=2)

    if "mean_reward_this_iter" in df:
        fig.add_trace(go.Bar(
            x=it, y=df["mean_reward_this_iter"],
            name="Mean this iter", marker_color=orange, opacity=0.5, showlegend=False,
        ), row=1, col=2)

    # --- Row 2: proxy quality ---
    if "proxy_r2" in df:
        r2 = df["proxy_r2"].replace([np.inf, -np.inf], np.nan)
        fig.add_trace(go.Scatter(
            x=it, y=r2,
            name="R²", line=dict(color="#55A868", width=2),
            mode="lines+markers", marker=dict(size=5),
        ), row=2, col=1)
        fig.add_hline(y=0, line=dict(color="grey", dash="dot", width=1), row=2, col=1)
        fig.add_hline(y=1, line=dict(color="grey", dash="dot", width=1), row=2, col=1)

    if "proxy_mae" in df:
        fig.add_trace(go.Scatter(
            x=it, y=df["proxy_mae"],
            name="MAE", line=dict(color="#C44E52", width=2),
            mode="lines+markers", marker=dict(size=5),
        ), row=2, col=2)

    # --- Row 3: oracle evals + time ---
    if "oracle_evals_total" in df:
        fig.add_trace(go.Scatter(
            x=it, y=df["oracle_evals_total"],
            name="Oracle evals", line=dict(color="#8172B3", width=2),
            mode="lines+markers", marker=dict(size=5),
        ), row=3, col=1)

    if "elapsed_sec" in df:
        fig.add_trace(go.Bar(
            x=it, y=df["elapsed_sec"],
            name="Elapsed (s)", marker_color="#937860", opacity=0.7,
        ), row=3, col=2)

    fig.update_layout(
        title=dict(text=f"AL run — <b>{run}</b>", font_size=16),
        height=800,
        legend=dict(bgcolor="rgba(0,0,0,0)", x=1.02, y=1),
        plot_bgcolor="white",
        paper_bgcolor="white",
        hovermode="x unified",
    )
    for row in range(1, 4):
        for col in range(1, 3):
            fig.update_xaxes(title_text="AL iteration", gridcolor="#eeeeee", row=row, col=col)
    fig.update_yaxes(gridcolor="#eeeeee")

    return fig




In [ ]:
plot_run("grid_results/run_138/al_metrics.csv")

In [ ]:

def compare_runs(
    names: list[str] = None,
) -> go.Figure:
    """
    Compare multiple AL runs side by side.

    Paths can be:
        - explicit list of CSV paths
        - a glob pattern string, e.g. "al_results/*/al_metrics.csv"

    Names default to the parent directory name of each CSV.

    Subplots
    --------
    Row 1 left  : best reward so far per run
    Row 1 right : mean reward so far per run
    Row 2 left  : proxy R² per run
    Row 2 right : oracle evals (efficiency — reward vs evals used)
    """
    from glob import glob
    paths = sorted(glob("grid_results/*/al_metrics.csv"))
    if not paths:
        raise ValueError("No files found at al_results/*/al_metrics.csv")
    if names is None:
        names = [Path(p).parent.name for p in paths]

    colors = _color_seq(len(paths))

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            "Best reward so far", "Mean reward so far",
            "Proxy R² over iterations", "Reward vs oracle evaluations",
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.10,
    )

    for (path, name, color) in zip(paths, names, colors):
        try:
            df = _load(path)
        except Exception as e:
            print(f"Skipping {path}: {e}")
            continue

        it = df["iteration"]
        style = dict(color=color, width=2)

        # best reward
        fig.add_trace(go.Scatter(
            x=it, y=df["best_reward_so_far"],
            name=name, line=style,
            mode="lines+markers", marker=dict(size=4),
            legendgroup=name,
        ), row=1, col=1)

        # mean reward
        fig.add_trace(go.Scatter(
            x=it, y=df["mean_reward_so_far"],
            name=name, line=style,
            mode="lines+markers", marker=dict(size=4),
            legendgroup=name, showlegend=False,
        ), row=1, col=2)

        # proxy R²
        if "proxy_r2" in df:
            r2 = df["proxy_r2"].replace([np.inf, -np.inf], np.nan)
            fig.add_trace(go.Scatter(
                x=it, y=r2,
                name=name, line=style,
                mode="lines+markers", marker=dict(size=4),
                legendgroup=name, showlegend=False,
            ), row=2, col=1)

        # reward vs oracle evals (sample efficiency)
        if "oracle_evals_total" in df:
            fig.add_trace(go.Scatter(
                x=df["oracle_evals_total"], y=df["best_reward_so_far"],
                name=name, line=style,
                mode="lines+markers", marker=dict(size=4),
                legendgroup=name, showlegend=False,
            ), row=2, col=2)

    fig.add_hline(y=0, line=dict(color="grey", dash="dot", width=1), row=2, col=1)

    fig.update_layout(
        title=dict(text=f"AL run comparison ({len(paths)} runs)", font_size=16),
        height=700,
        legend=dict(bgcolor="rgba(0,0,0,0)", x=1.02, y=1),
        plot_bgcolor="white",
        paper_bgcolor="white",
        hovermode="x unified",
    )
    fig.update_xaxes(title_text="AL iteration", gridcolor="#eeeeee", row=1, col=1)
    fig.update_xaxes(title_text="AL iteration", gridcolor="#eeeeee", row=1, col=2)
    fig.update_xaxes(title_text="AL iteration", gridcolor="#eeeeee", row=2, col=1)
    fig.update_xaxes(title_text="Oracle evaluations used", gridcolor="#eeeeee", row=2, col=2)
    fig.update_yaxes(gridcolor="#eeeeee")

    return fig

In [ ]:
compare_runs()

In [ ]:
def plot_grid_summary(grid: pd.DataFrame, save_dir: str = None):
    """Produce a compact set of aggregate plots from a grid search summary.

    Parameters
    ----------
    grid : pd.DataFrame
        Must contain columns:
        run, status, best_reward, mean_reward,
        sampling_strategy, acquisition, n_candidates_per_iter, n_init, seed
    save_dir : str or Path, optional
        If provided, figures are saved as PDFs in this directory.

    Returns
    -------
    dict of matplotlib Figure
    """
    import matplotlib.pyplot as plt

    # Keep only successful runs
    ok = grid[grid["status"] == "ok"].copy()
    if len(ok) == 0:
        print("[plot_grid_summary] No successful runs to plot.")
        return {}

    figs = {}

    # ── helper for bar plots with error bars ──
    def _bar(ax, df, x_label, y_label, title, hue=None, rot=0):
        if hue is None:
            grouped = df.groupby(x_label)["best_reward"]
            means = grouped.mean()
            stds = grouped.std().fillna(0)
            ax.bar(means.index, means.values, yerr=stds.values,
                   capsize=4, color="#4C72B0", edgecolor="white")
        else:
            # Grouped bar: one group per x_label, one bar per hue value
            pivot = df.pivot_table(index=x_label, columns=hue,
                                   values="best_reward", aggfunc="mean")
            err = df.pivot_table(index=x_label, columns=hue,
                                 values="best_reward", aggfunc="std").fillna(0)
            x_pos = range(len(pivot))
            n_hues = len(pivot.columns)
            width = 0.7 / n_hues
            colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
            for i, col in enumerate(pivot.columns):
                offset = (i - (n_hues - 1) / 2) * width
                ax.bar([p + offset for p in x_pos], pivot[col].values,
                       yerr=err[col].values if not err.empty else None,
                       width=width, label=str(col), capsize=3,
                       color=colors[i % len(colors)], edgecolor="white")
            ax.legend(title=hue)
        ax.set_xticks(range(len(ok[x_label].unique())))
        ax.set_xticklabels(sorted(ok[x_label].unique()), rotation=rot)
        ax.set_ylabel(y_label)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)

    # ── 1. Best reward by sampling strategy (averaged over everything) ──
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    _bar(ax1, ok, "sampling_strategy",
         "Best reward (mean ± std)", "Best reward by strategy")
    figs["grid_best_by_strategy"] = fig1

    # ── 2. Best reward by strategy × acquisition ──
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    _bar(ax2, ok, "sampling_strategy", "Best reward",
         "Best reward by strategy and acquisition", hue="acquisition", rot=15)
    figs["grid_best_strategy_acq"] = fig2

    # ── 3. Effect of initial dataset size ──
    fig3, ax3 = plt.subplots(figsize=(7, 4))
    _bar(ax3, ok, "n_init", "Best reward",
         "Effect of initial sample budget", hue="sampling_strategy")
    figs["grid_best_vs_n_init"] = fig3

    # ── 4. Effect of candidates per iteration ──
    fig4, ax4 = plt.subplots(figsize=(7, 4))
    _bar(ax4, ok, "n_candidates_per_iter", "Best reward",
         "Effect of candidates per iteration", hue="sampling_strategy")
    figs["grid_best_vs_n_candidates"] = fig4

    # ── 5. Top-10 runs ──
    top10 = ok.nlargest(10, "best_reward")
    fig5, ax5 = plt.subplots(figsize=(10, 5))
    colors = {"gflownet": "#4C72B0", "random": "#DD8452",
              "lhs": "#55A868", "grid": "#C44E52"}
    bar_colors = [colors.get(s, "#636363") for s in top10["sampling_strategy"]]
    bars = ax5.barh(range(len(top10)), top10["best_reward"].values,
                    color=bar_colors, edgecolor="white")
    ax5.set_yticks(range(len(top10)))
    ax5.set_yticklabels(top10["run"].values)
    ax5.invert_yaxis()
    ax5.set_xlabel("Best reward")
    ax5.set_title("Top-10 configurations")
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=s)
                       for s, c in colors.items()]
    ax5.legend(handles=legend_elements, title="Strategy")
    ax5.grid(axis="x", alpha=0.3)
    figs["grid_top10"] = fig5

    if save_dir:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        for name, fig in figs.items():
            fig.savefig(save_dir / f"{name}.pdf", bbox_inches="tight")
        print(f"[plot_grid_summary] Saved {len(figs)} figures → {save_dir}")

    return figs

In [ ]:
plot_grid_summary(grid)

## reset

In [11]:
all_metrics = attach_grid_metadata(df, grid)

In [12]:
all_metrics.head()

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,sampling_strategy,...,elapsed_sec,run,acquisition,n_candidates_per_iter,n_init,n_iterations,total_oracle_budget,seed,best_reward,mean_reward
0,1,17:17:16,70,70,20,0.953214,0.773339,0.208058,0.094107,random,...,6.6,run_0,top_k,20,50,15,350,123,0.968947,0.866844
1,2,17:17:18,90,90,20,0.953214,0.797323,0.569753,0.099253,random,...,2.5,run_0,top_k,20,50,15,350,123,0.968947,0.866844
2,3,17:17:19,110,110,20,0.968947,0.813842,0.675596,0.071394,random,...,0.4,run_0,top_k,20,50,15,350,123,0.968947,0.866844
3,4,17:17:19,130,130,20,0.968947,0.828889,0.718318,0.060543,random,...,0.4,run_0,top_k,20,50,15,350,123,0.968947,0.866844
4,5,17:17:20,150,150,20,0.968947,0.838112,0.627701,0.062788,random,...,0.4,run_0,top_k,20,50,15,350,123,0.968947,0.866844


In [13]:
metrics, grid = load_metrics(run)

In [14]:
metrics.head()

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,best_reward_this_iter,...,total_elapsed_sec,best_reward,mean_reward,total_oracle_budget,sampling_strategy,acquisition,seed,n_init,n_candidates_per_iter,n_iterations
0,1,17:17:16,70,70,20,0.953214,0.773339,0.208058,0.094107,0.953214,...,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15
1,2,17:17:18,90,90,20,0.953214,0.797323,0.569753,0.099253,0.951304,...,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15
2,3,17:17:19,110,110,20,0.968947,0.813842,0.675596,0.071394,0.968947,...,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15
3,4,17:17:19,130,130,20,0.968947,0.828889,0.718318,0.060543,0.950932,...,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15
4,5,17:17:20,150,150,20,0.968947,0.838112,0.627701,0.062788,0.962289,...,14.9,0.968947,0.866844,350,random,top_k,123,50,20,15


In [15]:
select_best_runs(metrics, metric="best_reward")

,iteration,time,n_samples,oracle_evals_total,oracle_evals_this_iter,best_reward_so_far,mean_reward_so_far,proxy_r2,proxy_mae,best_reward_this_iter,...,total_elapsed_sec,best_reward,mean_reward,total_oracle_budget,sampling_strategy,acquisition,seed,n_init,n_candidates_per_iter,n_iterations
45,1,23:24:52,70,70,20,0.987348,0.781351,0.600528,0.068155,0.987348,...,11.4,1.0,0.932578,350,genetic,diverse_top_k,456,50,20,15
46,2,23:24:52,90,90,20,0.987348,0.820528,0.656051,0.081237,0.975484,...,11.4,1.0,0.932578,350,genetic,diverse_top_k,456,50,20,15
47,3,23:24:53,110,110,20,0.993160,0.846150,0.632138,0.060151,0.993160,...,11.4,1.0,0.932578,350,genetic,diverse_top_k,456,50,20,15
48,4,23:24:54,130,130,20,0.998014,0.864462,0.729967,0.055114,0.998014,...,11.4,1.0,0.932578,350,genetic,diverse_top_k,456,50,20,15
49,5,23:24:54,150,150,20,1.000000,0.877836,0.630451,0.051532,1.000000,...,11.4,1.0,0.932578,350,genetic,diverse_top_k,456,50,20,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,11,17:18:47,310,310,10,1.000000,0.785822,0.784447,0.065783,0.945300,...,6.7,1.0,0.798122,350,random,diverse_top_k,456,200,10,15
191,12,17:18:47,320,320,10,1.000000,0.789684,0.779878,0.066589,0.950544,...,6.7,1.0,0.798122,350,random,diverse_top_k,456,200,10,15
192,13,17:18:48,330,330,10,1.000000,0.791846,0.778378,0.067077,0.945442,...,6.7,1.0,0.798122,350,random,diverse_top_k,456,200,10,15
193,14,17:18:48,340,340,10,1.000000,0.795242,0.797447,0.060427,0.943751,...,6.7,1.0,0.798122,350,random,diverse_top_k,456,200,10,15


In [ ]:
plot_reward_curves(metrics, metric="mean_reward_this_iter", title="Mean reward per iteration")

In [ ]:
aggregate_seeds(metrics, "mean_reward_this_iter", groupby=("sampling_strategy",)).groupby(["sampling_strategy", "iteration"]).agg(
    mean=("mean", "mean"),
    std=("std", "mean"),
    sem=("sem", "mean"),
    ci95=("ci95", "mean"),
)

mean       std       sem      ci95
sampling_strategy iteration                                        
genetic           1          0.940406  0.018510  0.010687  0.020947
                  2          0.948917  0.015087  0.008711  0.017073
                  3          0.959567  0.015905  0.009183  0.017999
                  4          0.960556  0.017057  0.009848  0.019302
                  5          0.966885  0.012003  0.006930  0.013582
...                               ...       ...       ...       ...
random            11         0.914151  0.009278  0.005357  0.010499
                  12         0.913194  0.007736  0.004466  0.008754
                  13         0.893757  0.020554  0.011867  0.023259
                  14         0.909748  0.010702  0.006179  0.012110
                  15         0.900494  0.014446  0.008340  0.016347

[90 rows x 4 columns]

In [ ]:
plot_seed_variance(metrics, metric="mean_reward_this_iter")

In [ ]:
plot_oracle_efficiency(metrics, metric="mean_reward_this_iter")

In [ ]:
plot_proxy_curves(metrics)

In [ ]:
plot_final_boxplot(metrics, metric="best_reward")

In [ ]:
plot_reward_heatmap(metrics, metric="best_reward")

In [ ]:
plot_budget_tradeoff(metrics, metric="mean_reward")

In [20]:
embedding, preprocessor, reducer = compute_global_embedding(
    method="pca"
)

In [21]:
sampled_dict_best = {}
for samp in df["sampling_strategy"].unique():
    run_id = best_runs(df).loc[best_runs(df)["sampling_strategy"] == samp, "run"].values[0]
    run_df = pd.read_csv(f"grid_results_2/{run_id}/dataset.csv")
    sampled_dict_best[samp] = prepare_run_embedding(run_df, preprocessor, reducer)

In [22]:
plot_sampling_grid(sampled_dict_best, cols=2, colorscale="Plasma")

In [ ]:
plot_sampling_grid(sampled_dict_best, cols=2, colorscale="Plasma")

In [ ]:
plot_exploration_dashboard(sampled_dict_best, preprocessor)

In [23]:
plot_peak_coverage(sampled_dict_best, "inverse")

In [24]:
plot_peak_coverage_heatmap(sampled_dict_best)

In [25]:
sampled_dict_all = {}
for samp in df["sampling_strategy"].unique():
    run_ids = df.loc[df["sampling_strategy"] == samp, "run"].unique()
    pooled = pd.concat(
        [pd.read_csv(f"grid_results_2/{r}/dataset.csv") for r in run_ids],
        ignore_index=True,
    )
    sampled_dict_all[samp] = prepare_run_embedding(pooled, preprocessor, reducer)
    

In [26]:
plot_exploration_dashboard(sampled_dict_all, preprocessor)

In [27]:
plot_peak_coverage(sampled_dict_all, 'inverse')

In [28]:
plot_peak_coverage_heatmap(sampled_dict_all)